### 01. Implementing modelling

First we want to create a simple linear regression to serve as our baseline.

In [ ]:
# First we got to do some imports (I'm just grabbing the ones from 02 for now)
from pathlib import Path
import platform
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.linear_model import LinearRegression


# Random seed for reproducibility
np.random.seed(42)

df = pd.read_csv("data/ecowas_df_synthetic_full.csv")

In [28]:
df.columns

Index(['Unnamed: 0', 'country', 'year', 'disorder_Demonstrations',
       'disorder_Political violence',
       'disorder_Political violence; Demonstrations',
       'disorder_Strategic developments', 'event_Battles',
       'event_Explosions/Remote violence', 'event_Protests', 'event_Riots',
       'event_Strategic developments', 'event_Violence against civilians',
       'perpetrator_Civilians', 'perpetrator_External/Other forces',
       'perpetrator_Identity militia', 'perpetrator_Political militia',
       'perpetrator_Protesters', 'perpetrator_Rebel group',
       'perpetrator_Rioters', 'perpetrator_State forces', 'target_Civilians',
       'target_External/Other forces', 'target_Identity militia',
       'target_Political militia', 'target_Protesters', 'target_Rebel group',
       'target_Rioters', 'target_State forces', 'fatalities', 'iso3_o',
       'iso3_d', 'distw_harmonic', 'distw_arithmetic', 'dist', 'distcap',
       'contig', 'diplo_disagreement', 'comlang_off', 'comlang

In [29]:
all_targetv = ["tradeflow_baci", "tradeflow_comtrade_o", "tradeflow_comtrade_d", 
                "tradeflow_imf_o", "tradeflow_imf_d", "combined_trade_baci",]

In [30]:
train_df = df[df["year"] <= 2016]
test_df  = df[df["year"] > 2016]

In [62]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error

### Log/Linear Regression on Economic/Gravity Data

This will serve as our primary baseline. If the machine learning model does not improve on this, then it does not add predictive power beyond simple Gravity/trade fundamentals

In [65]:
legal_target_vars = {"tradeflow_baci", 'tradeflow_comtrade_o', 'tradeflow_comtrade_d', 'tradeflow_imf_o', 'tradeflow_imf_d', "combined_trade_baci"}
required_columns = {"gdp_o", "gdp_d", "distw_arithmetic"}
required_columnsv2 = {"gdp_o", "gdp_d", "dist"}

def baseline_linear(df: pd.DataFrame, target_variable: str, columns: list, year_split = 2016):
    '''
    The main function for creating our regression

    
    Input:
        df: pd.DataFrame -
            The main dataframe to work with.
        target_variable: str -
            The column which we are inferring for. 
        columns: list -
            A list of column elements that are included in the model. NOTE: Always include the basic Gravity columns, other we raise an error
        year_split: int -
            The year where we split between train and test. Defaults to 2016
    '''
    columns = columns.copy()
    train_df_full = df[df["year"] <= year_split].copy()
    test_df_full = df[df["year"] > year_split].copy()
    
    missing_columns = required_columns - set(columns)
    if missing_columns:
        raise ValueError(f"Function columns MUST include all of the following columns from dataframe: {required_columns}.")
    
    # We check for validity in input
    if target_variable not in legal_target_vars:    
        raise ValueError(f"Invalid target variable '{target_variable}'. \n"
                         f"Target variable must be one of {legal_target_vars}.")
    
    


    all_needed = columns + [target_variable]
    train_df = train_df_full[all_needed].copy()
    test_df = test_df_full[all_needed].copy()

    train_df = train_df.dropna()
    test_df = test_df.dropna()


    train_df[f"{target_variable}_log_trade"] = np.log(train_df[target_variable])
    test_df[f"{target_variable}_log_trade"]  = np.log(test_df[target_variable])

    train_df["log_gdp_o"] = np.log(train_df["gdp_o"])
    train_df["log_gdp_d"] = np.log(train_df["gdp_d"])
    train_df["log_distw"]  = np.log(train_df["distw_arithmetic"])

    test_df["log_gdp_o"] = np.log(test_df["gdp_o"])
    test_df["log_gdp_d"] = np.log(test_df["gdp_d"])
    test_df["log_distw"]  = np.log(test_df["distw_arithmetic"])

    update_list_holder = ["gdp_o", "log_gdp_o", "gdp_d", "log_gdp_d", "distw_arithmetic", "log_distw"]
    for i in range(0, len(update_list_holder), 2):
        columns.remove(update_list_holder[i])
        columns.append(update_list_holder[i+1])
    

    model_columns = columns + [f"{target_variable}_log_trade"]

    train_df_model = train_df[model_columns].dropna()
    test_df_model = test_df[model_columns].dropna()

    X_train = train_df_model[columns]
    y_train = train_df_model[f"{target_variable}_log_trade"]

    X_test = test_df_model[columns]
    y_test = test_df_model[f"{target_variable}_log_trade"]


    model = LinearRegression()
    model.fit(X_train, y_train)


    # Finally we can get some workable metrics from this regression:
    y_pred = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)

    print("Out-of-sample RMSE:", rmse)
    print("Out-of-sample R²:", r2)



    groups = train_df_full.loc[train_df_model.index, "dyad"]
    X = sm.add_constant(X_train)
    ols = sm.OLS(y_train, X).fit(cov_type="cluster",
                                cov_kwds={"groups": groups},)

    print(ols.summary())


    return {
    "model_name": target_variable,
    "n_train": len(X_train),
    "n_test": len(X_test),
    "rmse": rmse,
    "mae": mae,
    "r2": r2,
}

    


In [66]:
results = []

for target in legal_target_vars:
    print(f"\n{'='*60}")
    print(f"Target: {target}")
    print(f"{'='*60}")
    r = baseline_linear(df, target, columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'])
    results.append(r)

comparison = pd.DataFrame(
    [
        {
            "Model"   : r["model_name"],
            "N train" : r["n_train"],
            "N test"  : r["n_test"],
            "RMSE"    : round(r["rmse"], 4),
            "R²"      : round(r["r2"],   4),
            "mae"     : round(r["mae"],  4)
        }
        for r in results
    ]
).set_index("Model")

print("\n" + "="*80)
print("   SAMMENLIGNING — OUT-OF-SAMPLE PERFORMANCE (år > 2016)")
print("="*80)
print(comparison.to_string())
print("="*80 + "\n")



Target: tradeflow_baci
Out-of-sample RMSE: 1.623279712343434
Out-of-sample R²: 0.3368842466059815
                               OLS Regression Results                               
Dep. Variable:     tradeflow_baci_log_trade   R-squared:                       0.360
Model:                                  OLS   Adj. R-squared:                  0.359
Method:                       Least Squares   F-statistic:                     39.37
Date:                      Tue, 05 May 2026   Prob (F-statistic):           3.91e-23
Time:                              14:31:04   Log-Likelihood:                -6337.2
No. Observations:                      3458   AIC:                         1.269e+04
Df Residuals:                          3451   BIC:                         1.273e+04
Df Model:                                 6                                         
Covariance Type:                    cluster                                         
                    coef    std err          z     

/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 6, but rank is 5
  warnings.warn('covariance of constraints does not have full '


### Linear Regression on Conflict Data

We will try to establish a baseline using the conflict data as well. This should be a combination of the trade model above along with a subset of conflict data (as too many regressors will break the model) 

In [91]:
import contextlib, io

def find_best_conflict_combo(df, target_variable, base_columns=None, year_split=2016):
    if base_columns is None:
        base_columns = ["gdp_o", "gdp_d", "distw_arithmetic", "contig", "comlang_off", "comlang_ethno"]

    # Build a "total_conflicts" column as the sum of all conflict-related columns
    conflict_prefixes = ["disorder_", "event_", "perpetrator_", "target_"]
    all_conflict_cols = [c for c in df.columns if any(c.startswith(p) for p in conflict_prefixes)] + ["fatalities"]
    df = df.copy()
    df["total_conflicts"] = df[all_conflict_cols].sum(axis=1)

    conflict_groups = {
        "disorder":        [c for c in df.columns if c.startswith("disorder_")],
        "event":           [c for c in df.columns if c.startswith("event_")],
        "perpetrator":     [c for c in df.columns if c.startswith("perpetrator_")],
        "target":          [c for c in df.columns if c.startswith("target_")],
        "fatalities":      ["fatalities"],
        "total_conflicts": ["total_conflicts"],
    }

    group_names = list(conflict_groups.keys())
    results = []

    for r in range(1, len(group_names) + 1):
        for combo in combinations(group_names, r):
            extra_cols = []
            for g in combo:
                extra_cols.extend(conflict_groups[g])

            columns = base_columns + extra_cols
            label = " + ".join(combo)

            try:
                with contextlib.redirect_stdout(io.StringIO()):
                    res = baseline_linear(df, target_variable, columns, year_split)
                res["model_name"] = label
                results.append(res)
            except Exception as e:
                print(f"  ⚠ {label} failed: {e}")

    results_df = pd.DataFrame(results).sort_values("mae", ascending=True).reset_index(drop=True)

    display_df = results_df.copy()
    display_df["rmse"] = display_df["rmse"].round(4)
    display_df["mae"]  = display_df["mae"].round(4)
    display_df["r2"]   = display_df["r2"].round(4)

    print("\n" + "="*90)
    print(f"   RANKING — {target_variable} (best MAE first)")
    print("="*90)
    print(display_df.to_string())
    print("="*90 + "\n")

    return results_df


In [100]:
import contextlib, io
from itertools import combinations

def find_best_conflict_combo(df, target_variable, base_columns=None, year_split=2016):
    if base_columns is None:
        base_columns = ["gdp_o", "gdp_d", "distw_arithmetic", "contig", "comlang_off", "comlang_ethno"]

    conflict_prefixes = ["disorder_", "event_", "perpetrator_", "target_"]
    all_conflict_cols = [c for c in df.columns if any(c.startswith(p) for p in conflict_prefixes)] + ["fatalities"]
    df = df.copy()
    df["total_conflicts"] = df[all_conflict_cols].sum(axis=1)

    conflict_groups = {
        "disorder":    [c for c in df.columns if c.startswith("disorder_")],
        "event":       [c for c in df.columns if c.startswith("event_")],
        "perpetrator": [c for c in df.columns if c.startswith("perpetrator_")],
        "target":      [c for c in df.columns if c.startswith("target_")],
    }

    base_groups = list(conflict_groups.keys())
    results = []

    # Build all combos to test
    combos_to_test = []

    # 1) All non-empty subsets of the 4 base groups (15 combos)
    for r in range(1, len(base_groups) + 1):
        for combo in combinations(base_groups, r):
            extra = []
            for g in combo:
                extra.extend(conflict_groups[g])
            combos_to_test.append((" + ".join(combo), extra))

    # 2) All 4 groups + fatalities
    all_cols = [c for g in base_groups for c in conflict_groups[g]] + ["fatalities"]
    combos_to_test.append(("disorder + event + perpetrator + target + fatalities", all_cols))

    # 3) Fatalities alone
    combos_to_test.append(("fatalities", ["fatalities"]))

    # 4) Total conflicts alone
    combos_to_test.append(("total_conflicts", ["total_conflicts"]))

    # 5) Total conflicts + fatalities
    combos_to_test.append(("total_conflicts + fatalities", ["total_conflicts", "fatalities"]))


    # Run all combos
    for label, extra_cols in combos_to_test:
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                res = baseline_linear(df, target_variable, base_columns + extra_cols, year_split)
            res["model_name"] = label
            results.append(res)
        except Exception as e:
            print(f"  ⚠ {label} failed: {e}")

    results_df = pd.DataFrame(results).sort_values("mae", ascending=True).reset_index(drop=True)

    display_df = results_df.copy()
    display_df["rmse"] = display_df["rmse"].round(4)
    display_df["mae"]  = display_df["mae"].round(4)
    display_df["r2"]   = display_df["r2"].round(4)

    print("\n" + "="*100)
    print(f"   RANKING — {target_variable} (best MAE first)")
    print("="*100)
    print(display_df.to_string())
    print("="*100 + "\n")

    return results_df


In [93]:
results_df = find_best_conflict_combo(df, "tradeflow_baci")


/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 16, but rank is 14
  warnings.warn('covariance of constraints does not have full '
/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 18, but rank is 17
  warnings.warn('covariance of constraints does not have full '
/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 20, but rank is 19
  warnings.warn('covariance of constraints does not have full '
/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 24, but rank is 22
  warnin


   RANKING — tradeflow_baci (best MAE first)
                                              model_name  n_train  n_test    rmse     mae      r2
0                           total_conflicts + fatalities     3458     910  1.6235  1.2787  0.3367
1                                             fatalities     3458     910  1.6282  1.2867  0.3329
2                                        total_conflicts     3458     910  1.6458  1.3107  0.3184
3                                               disorder     3458     910  1.7474  1.3589  0.2316
4                                                  event     3458     910  1.7351  1.3749  0.2424
5                                       disorder + event     3458     910  1.7744  1.3907  0.2077
6                                    event + perpetrator     3458     910  2.1387  1.5417 -0.1510
7                         disorder + event + perpetrator     3458     910  2.2653  1.5533 -0.2914
8                                                 target     3458     91

In [94]:
results_df = find_best_conflict_combo(df, "tradeflow_comtrade_o")

/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 16, but rank is 14
  warnings.warn('covariance of constraints does not have full '
/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 18, but rank is 17
  warnings.warn('covariance of constraints does not have full '
/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 20, but rank is 19
  warnings.warn('covariance of constraints does not have full '
/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 24, but rank is 22
  warnin


   RANKING — tradeflow_comtrade_o (best MAE first)
                                              model_name  n_train  n_test    rmse     mae      r2
0                           total_conflicts + fatalities     3458     910  1.4045  1.0779  0.3679
1                                             fatalities     3458     910  1.4169  1.0989  0.3567
2                                        total_conflicts     3458     910  1.4312  1.1173  0.3437
3                                               disorder     3458     910  1.4816  1.1604  0.2966
4                                                  event     3458     910  1.5418  1.2322  0.2383
5                                            perpetrator     3458     910  1.5840  1.2793  0.1961
6                                       disorder + event     3458     910  1.6405  1.2877  0.1377
7                                   perpetrator + target     3458     910  1.7058  1.3057  0.0677
8                                    event + perpetrator     3458 

/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 32, but rank is 29
  warnings.warn('covariance of constraints does not have full '
/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 33, but rank is 30
  warnings.warn('covariance of constraints does not have full '


In [95]:
results_df = find_best_conflict_combo(df, "tradeflow_comtrade_d")

/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 16, but rank is 14
  warnings.warn('covariance of constraints does not have full '
/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 20, but rank is 19
  warnings.warn('covariance of constraints does not have full '
/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 24, but rank is 21
  warnings.warn('covariance of constraints does not have full '
/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 24, but rank is 22
  warnin


   RANKING — tradeflow_comtrade_d (best MAE first)
                                              model_name  n_train  n_test    rmse     mae      r2
0                                             fatalities     3458     910  1.4956  1.1745  0.2737
1                           total_conflicts + fatalities     3458     910  1.4965  1.1757  0.2729
2                                        total_conflicts     3458     910  1.5164  1.2014  0.2533
3                                       disorder + event     3458     910  1.5310  1.2169  0.2390
4                                                  event     3458     910  1.5532  1.2414  0.2167
5                                               disorder     3458     910  1.5679  1.2442  0.2018
6                              disorder + event + target     3458     910  1.9834  1.4041 -0.2774
7                                         event + target     3458     910  1.9229  1.4278 -0.2006
8                         disorder + event + perpetrator     3458 

In [96]:
results_df = find_best_conflict_combo(df, "tradeflow_imf_o")
print(results_df.to_string())


/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 16, but rank is 14
  warnings.warn('covariance of constraints does not have full '
/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 18, but rank is 17
  warnings.warn('covariance of constraints does not have full '
/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 20, but rank is 19
  warnings.warn('covariance of constraints does not have full '
/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 24, but rank is 22
  warnin


   RANKING — tradeflow_imf_o (best MAE first)
                                              model_name  n_train  n_test    rmse     mae      r2
0                           total_conflicts + fatalities     3458     910  1.6702  1.3084  0.2799
1                                             fatalities     3458     910  1.6736  1.3159  0.2770
2                                        total_conflicts     3458     910  1.6839  1.3338  0.2681
3                                               disorder     3458     910  1.7125  1.3579  0.2430
4                                                  event     3458     910  1.7062  1.3647  0.2486
5                                       disorder + event     3458     910  1.7268  1.3791  0.2303
6                                    event + perpetrator     3458     910  1.8692  1.4403  0.0981
7                         disorder + event + perpetrator     3458     910  1.9643  1.4546  0.0040
8                                                 target     3458     9

In [97]:
results_df = find_best_conflict_combo(df, "tradeflow_imf_d")
print(results_df.to_string())


/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 16, but rank is 14
  warnings.warn('covariance of constraints does not have full '
/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 20, but rank is 19
  warnings.warn('covariance of constraints does not have full '
/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 24, but rank is 21
  warnings.warn('covariance of constraints does not have full '
/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 24, but rank is 22
  warnin


   RANKING — tradeflow_imf_d (best MAE first)
                                              model_name  n_train  n_test    rmse     mae      r2
0                           total_conflicts + fatalities     3458     910  1.6628  1.3554  0.2177
1                                             fatalities     3458     910  1.6743  1.3697  0.2068
2                                        total_conflicts     3458     910  1.6893  1.3862  0.1926
3                                       disorder + event     3458     910  1.7370  1.4103  0.1463
4                                                  event     3458     910  1.7520  1.4197  0.1316
5                                               disorder     3458     910  1.7600  1.4235  0.1236
6                         disorder + event + perpetrator     3458     910  2.2685  1.6058 -0.4560
7                              disorder + event + target     3458     910  2.3635  1.6213 -0.5805
8                                    event + perpetrator     3458     9

/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 32, but rank is 29
  warnings.warn('covariance of constraints does not have full '
/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 33, but rank is 30
  warnings.warn('covariance of constraints does not have full '


In [98]:
results_df = find_best_conflict_combo(df, "combined_trade_baci")
print(results_df.to_string())


/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 16, but rank is 14
  warnings.warn('covariance of constraints does not have full '
/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 18, but rank is 17
  warnings.warn('covariance of constraints does not have full '
/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 20, but rank is 19
  warnings.warn('covariance of constraints does not have full '
/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 24, but rank is 22
  warnin


   RANKING — combined_trade_baci (best MAE first)
                                              model_name  n_train  n_test    rmse     mae      r2
0                                             fatalities     3458     910  1.1677  0.8245  0.4812
1                           total_conflicts + fatalities     3458     910  1.1715  0.8311  0.4778
2                                        total_conflicts     3458     910  1.1821  0.8459  0.4683
3                                               disorder     3458     910  1.2812  0.8985  0.3755
4                                                  event     3458     910  1.2974  0.9362  0.3595
5                                            perpetrator     3458     910  1.3731  0.9534  0.2826
6                                    event + perpetrator     3458     910  1.3764  0.9668  0.2791
7                                       disorder + event     3458     910  1.3793  0.9703  0.2761
8                         disorder + event + perpetrator     3458  

/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 33, but rank is 31
  warnings.warn('covariance of constraints does not have full '


#### FE Implementation (Anderson TABLE of FIXED EFFECTS and PARAMETERS)

In [99]:
def baseline_linear_fe(df: pd.DataFrame, target_variable: str, columns: list, year_split=2016):
    '''
    OLS gravity baseline with exporter and importer fixed effects.
    '''
    columns = columns.copy()
    train_df_full = df[df["year"] <= year_split].copy()
    test_df_full  = df[df["year"] > year_split].copy()

    missing_columns = required_columns - set(columns)
    if missing_columns:
        raise ValueError(f"Function columns MUST include all of: {required_columns}.")

    if target_variable not in legal_target_vars:
        raise ValueError(f"Invalid target variable '{target_variable}'. Must be one of {legal_target_vars}.")

    all_needed = columns + [target_variable]
    train_df = train_df_full[all_needed].copy()
    test_df  = test_df_full[all_needed].copy()

    train_df = train_df.dropna()
    test_df  = test_df.dropna()

    # Log-transform target
    train_df[f"{target_variable}_log_trade"] = np.log(train_df[target_variable])
    test_df[f"{target_variable}_log_trade"]  = np.log(test_df[target_variable])

    # Log-transform gravity variables
    train_df["log_gdp_o"] = np.log(train_df["gdp_o"])
    train_df["log_gdp_d"] = np.log(train_df["gdp_d"])
    train_df["log_distw"] = np.log(train_df["distw_arithmetic"])

    test_df["log_gdp_o"] = np.log(test_df["gdp_o"])
    test_df["log_gdp_d"] = np.log(test_df["gdp_d"])
    test_df["log_distw"] = np.log(test_df["distw_arithmetic"])

    # Replace raw gravity vars with log versions in columns list
    update_list_holder = ["gdp_o", "log_gdp_o", "gdp_d", "log_gdp_d", "distw_arithmetic", "log_distw"]
    for i in range(0, len(update_list_holder), 2):
        columns.remove(update_list_holder[i])
        columns.append(update_list_holder[i + 1])

    # --- Add exporter and importer fixed effects ---
    for fe_col in ["iso3_o", "iso3_d"]:
        dummies_train = pd.get_dummies(
            train_df_full.loc[train_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float
        )
        dummies_test = pd.get_dummies(
            test_df_full.loc[test_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float
        )
        # Align test columns to train (handles missing categories)
        dummies_test = dummies_test.reindex(columns=dummies_train.columns, fill_value=0)

        train_df = pd.concat([train_df, dummies_train], axis=1)
        test_df  = pd.concat([test_df, dummies_test], axis=1)
        columns.extend(dummies_train.columns.tolist())

    # --- Build model matrices ---
    model_columns = columns + [f"{target_variable}_log_trade"]
    train_df_model = train_df[model_columns].dropna()
    test_df_model  = test_df[model_columns].dropna()

    X_train = train_df_model[columns]
    y_train = train_df_model[f"{target_variable}_log_trade"]
    X_test  = test_df_model[columns]
    y_test  = test_df_model[f"{target_variable}_log_trade"]

    # --- Fit OLS ---
    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    rmse = root_mean_squared_error(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    print("Out-of-sample RMSE:", rmse)
    print("Out-of-sample MAE:", mae)
    print("Out-of-sample R²:", r2)

    # --- Clustered standard errors ---
    groups = train_df_full.loc[train_df_model.index, "dyad"]
    X = sm.add_constant(X_train)
    ols = sm.OLS(y_train, X).fit(cov_type="cluster", cov_kwds={"groups": groups})
    print(ols.summary())

    return {
        "model_name": target_variable,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
    }


In [103]:
import contextlib, io
from itertools import combinations

def find_best_conflict_combo(df, target_variable, base_columns=None, year_split=2016):
    if base_columns is None:
        base_columns = ["gdp_o", "gdp_d", "distw_arithmetic", "contig", "comlang_off", "comlang_ethno"]

    conflict_prefixes = ["disorder_", "event_", "perpetrator_", "target_"]
    all_conflict_cols = [c for c in df.columns if any(c.startswith(p) for p in conflict_prefixes)] + ["fatalities"]
    df = df.copy()
    df["total_conflicts"] = df[all_conflict_cols].sum(axis=1)

    conflict_groups = {
        "disorder":    [c for c in df.columns if c.startswith("disorder_")],
        "event":       [c for c in df.columns if c.startswith("event_")],
        "perpetrator": [c for c in df.columns if c.startswith("perpetrator_")],
        "target":      [c for c in df.columns if c.startswith("target_")],
    }

    base_groups = list(conflict_groups.keys())
    results = []

    # Build all combos to test
    combos_to_test = []

    # 1) All non-empty subsets of the 4 base groups (15 combos)
    for r in range(1, len(base_groups) + 1):
        for combo in combinations(base_groups, r):
            extra = []
            for g in combo:
                extra.extend(conflict_groups[g])
            combos_to_test.append((" + ".join(combo), extra))

    # 2) All 4 groups + fatalities
    all_cols = [c for g in base_groups for c in conflict_groups[g]] + ["fatalities"]
    combos_to_test.append(("disorder + event + perpetrator + target + fatalities", all_cols))

    # 3) Fatalities alone
    combos_to_test.append(("fatalities", ["fatalities"]))

    # 4) Total conflicts alone
    combos_to_test.append(("total_conflicts", ["total_conflicts"]))

    # 5) Total conflicts + fatalities
    combos_to_test.append(("total_conflicts + fatalities", ["total_conflicts", "fatalities"]))


    # Run all combos
    for label, extra_cols in combos_to_test:
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                res = baseline_linear_fe(df, target_variable, base_columns + extra_cols, year_split)
            res["model_name"] = label
            results.append(res)
        except Exception as e:
            print(f"  ⚠ {label} failed: {e}")

    results_df = pd.DataFrame(results).sort_values("mae", ascending=True).reset_index(drop=True)

    display_df = results_df.copy()
    display_df["rmse"] = display_df["rmse"].round(4)
    display_df["mae"]  = display_df["mae"].round(4)
    display_df["r2"]   = display_df["r2"].round(4)

    print("\n" + "="*100)
    print(f"   RANKING — {target_variable} (best MAE first)")
    print("="*100)
    print(display_df.to_string())
    print("="*100 + "\n")

    return results_df


In [104]:
results_df = find_best_conflict_combo(df, "combined_trade_baci")


/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 36, but rank is 23
  warnings.warn('covariance of constraints does not have full '
/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 38, but rank is 25
  warnings.warn('covariance of constraints does not have full '
/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 40, but rank is 27
  warnings.warn('covariance of constraints does not have full '
/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 40, but rank is 27
  warnin


   RANKING — combined_trade_baci (best MAE first)
                                              model_name  n_train  n_test    rmse     mae      r2
0                                            perpetrator     3458     910  1.0799  0.8216  0.5563
1                                             fatalities     3458     910  1.0736  0.8259  0.5614
2                           total_conflicts + fatalities     3458     910  1.0757  0.8281  0.5597
3                                 disorder + perpetrator     3458     910  1.0916  0.8355  0.5466
4                                    event + perpetrator     3458     910  1.0980  0.8373  0.5412
5                                        total_conflicts     3458     910  1.0862  0.8385  0.5510
6   disorder + event + perpetrator + target + fatalities     3458     910  1.1076  0.8391  0.5332
7                        disorder + perpetrator + target     3458     910  1.1093  0.8468  0.5317
8                disorder + event + perpetrator + target     3458  

/Users/zenrehda/anaconda3/lib/python3.11/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 34, but rank is 21
  warnings.warn('covariance of constraints does not have full '


### PPML Baseline
A PPML baseline that corresponds to best practice and also adheres to the same principles as the sampling of our synthetic data


In [107]:
def baseline_ppml_fe(
    df: pd.DataFrame,
    target_variable: str,
    columns: list,
    year_split: int = 2016
):
    """
    PPML gravity baseline with exporter and importer fixed effects.
    """

    # --- Defensive copy ---
    columns = columns.copy()

    # --- Split sample ---
    train_df_full = df[df["year"] <= year_split].copy()
    test_df_full  = df[df["year"] > year_split].copy()

    # --- Column checks ---
    missing_columns = required_columns - set(columns)
    if missing_columns:
        raise ValueError(f"Function columns MUST include all of: {required_columns}")

    if target_variable not in legal_target_vars:
        raise ValueError(f"Invalid target variable '{target_variable}'. Must be one of {legal_target_vars}.")

    # --- Subset and drop NA ---
    all_needed = columns + [target_variable]
    train_df = train_df_full[all_needed].dropna().copy()
    test_df  = test_df_full[all_needed].dropna().copy()

    # --- Log-transform gravity covariates ---
    train_df["log_gdp_o"] = np.log(train_df["gdp_o"])
    train_df["log_gdp_d"] = np.log(train_df["gdp_d"])
    train_df["log_distw"] = np.log(train_df["distw_arithmetic"])

    test_df["log_gdp_o"] = np.log(test_df["gdp_o"])
    test_df["log_gdp_d"] = np.log(test_df["gdp_d"])
    test_df["log_distw"] = np.log(test_df["distw_arithmetic"])

    # --- Replace raw gravity variables with logs ---
    update_list_holder = [
        "gdp_o", "log_gdp_o",
        "gdp_d", "log_gdp_d",
        "distw_arithmetic", "log_distw"
    ]
    for i in range(0, len(update_list_holder), 2):
        columns.remove(update_list_holder[i])
        columns.append(update_list_holder[i + 1])

    # --- Add exporter and importer fixed effects ---
    for fe_col in ["iso3_o", "iso3_d"]:
        dummies_train = pd.get_dummies(
            train_df_full.loc[train_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float
        )
        dummies_test = pd.get_dummies(
            test_df_full.loc[test_df.index, fe_col], prefix=fe_col, drop_first=True, dtype=float
        )
        dummies_test = dummies_test.reindex(columns=dummies_train.columns, fill_value=0)

        train_df = pd.concat([train_df, dummies_train], axis=1)
        test_df  = pd.concat([test_df, dummies_test], axis=1)
        columns.extend(dummies_train.columns.tolist())

    # --- Model matrices ---
    X_train = sm.add_constant(train_df[columns])
    y_train = train_df[target_variable]

    X_test = sm.add_constant(test_df[columns])
    y_test = test_df[target_variable]

    # --- PPML estimation ---
    ppml_model = sm.GLM(
        y_train,
        X_train,
        family=sm.families.Poisson(sm.families.links.Log())
    )

    ppml_results = ppml_model.fit(
        cov_type="cluster",
        cov_kwds={"groups": train_df_full.loc[train_df.index, "dyad"]}
    )

    # --- Predictions in levels ---
    y_pred = ppml_results.predict(X_test)

    # --- Performance metrics ---
    rmse = root_mean_squared_error(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    print("Out-of-sample RMSE (levels):", rmse)
    print("Out-of-sample MAE (levels):", mae)
    print("Out-of-sample R² (levels):", r2)
    print(ppml_results.summary())

    return {
        "model_name": target_variable,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
    }


In [108]:
target_variable = baseline_ppml_fe(df, "combined_trade_baci", columns=["gdp_o", "gdp_d", "distw_arithmetic", "contig",'comlang_off', 'comlang_ethno'])

Out-of-sample RMSE (levels): 264197.5222478702
Out-of-sample MAE (levels): 110332.71285375982
Out-of-sample R² (levels): 0.28572807576781856
                  Generalized Linear Model Regression Results                  
Dep. Variable:     combined_trade_baci   No. Observations:                 3458
Model:                             GLM   Df Residuals:                     3425
Model Family:                  Poisson   Df Model:                           32
Link Function:                     Log   Scale:                          1.0000
Method:                           IRLS   Log-Likelihood:            -9.7379e+07
Date:                 Wed, 06 May 2026   Deviance:                   1.9472e+08
Time:                         12:38:44   Pearson chi2:                 3.57e+08
No. Iterations:                      8   Pseudo R-squ. (CS):              1.000
Covariance Type:               cluster                                         
                    coef    std err          z      P>|z|  

In [111]:
import contextlib, io
from itertools import combinations

def find_best_conflict_combo(df, target_variable, base_columns=None, year_split=2016):
    if base_columns is None:
        base_columns = ["gdp_o", "gdp_d", "distw_arithmetic", "contig", "comlang_off", "comlang_ethno"]

    conflict_prefixes = ["disorder_", "event_", "perpetrator_", "target_"]
    all_conflict_cols = [c for c in df.columns if any(c.startswith(p) for p in conflict_prefixes)] + ["fatalities"]
    df = df.copy()
    df["total_conflicts"] = df[all_conflict_cols].sum(axis=1)

    conflict_groups = {
        "disorder":    [c for c in df.columns if c.startswith("disorder_")],
        "event":       [c for c in df.columns if c.startswith("event_")],
        "perpetrator": [c for c in df.columns if c.startswith("perpetrator_")],
        "target":      [c for c in df.columns if c.startswith("target_")],
    }

    base_groups = list(conflict_groups.keys())
    results = []

    # Build all combos to test
    combos_to_test = []

    # 1) All non-empty subsets of the 4 base groups (15 combos)
    for r in range(1, len(base_groups) + 1):
        for combo in combinations(base_groups, r):
            extra = []
            for g in combo:
                extra.extend(conflict_groups[g])
            combos_to_test.append((" + ".join(combo), extra))

    # 2) All 4 groups + fatalities
    all_cols = [c for g in base_groups for c in conflict_groups[g]] + ["fatalities"]
    combos_to_test.append(("disorder + event + perpetrator + target + fatalities", all_cols))

    # 3) Fatalities alone
    combos_to_test.append(("fatalities", ["fatalities"]))

    # 4) Total conflicts alone
    combos_to_test.append(("total_conflicts", ["total_conflicts"]))

    # 5) Total conflicts + fatalities
    combos_to_test.append(("total_conflicts + fatalities", ["total_conflicts", "fatalities"]))


    # Run all combos
    for label, extra_cols in combos_to_test:
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                res = baseline_ppml_fe(df, target_variable, base_columns + extra_cols, year_split)
            res["model_name"] = label
            results.append(res)
        except Exception as e:
            print(f"  ⚠ {label} failed: {e}")

    results_df = pd.DataFrame(results).sort_values("mae", ascending=True).reset_index(drop=True)

    display_df = results_df.copy()
    display_df["rmse"] = display_df["rmse"].round(4)
    display_df["mae"]  = display_df["mae"].round(4)
    display_df["r2"]   = display_df["r2"].round(4)

    print("\n" + "="*100)
    print(f"   RANKING — {target_variable} (best MAE first)")
    print("="*100)
    print(display_df.to_string())
    print("="*100 + "\n")

    return results_df


In [112]:
results_df = find_best_conflict_combo(df, "tradeflow_baci")


   RANKING — tradeflow_baci (best MAE first)
                                              model_name  n_train  n_test         rmse         mae      r2
0                              disorder + event + target     3458     910  182090.5384  61531.6812  0.1745
1                                       disorder + event     3458     910  169967.6156  61876.9887  0.2808
2                                      disorder + target     3458     910  184013.7465  61990.7064  0.1570
3                                                  event     3458     910  171317.8117  62096.3700  0.2693
4                                                 target     3458     910  188047.5267  62963.9021  0.1196
5                                               disorder     3458     910  176624.7340  62981.9349  0.2233
6                                         event + target     3458     910  187475.9267  63106.4401  0.1250
7                         disorder + event + perpetrator     3458     910  178541.2640  63697.5154